# Dynamic Programming and Backtracking Lab

```{contents}
:local:
:depth: 2
```

Use a purchasing workflow to compare unlimited pack quantities with one-use invoice adjustments. Verify objective values and witnesses independently. Download the complete implementations and reference checks: [PlanningChecks.cs](../../materials/24/PlanningChecks.cs).

## Verify an Optimum and Its Witness

```{index} Verify an Optimum and Its Witness
```

For pack sizes 1,3,4 and quantity six, largest-first makes 4+1+1. Dynamic programming returns a two-pack plan. Independently check allowed sizes, witness count, and total, rather than trusting only a printed minimum.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={1,3,4};int target=6;
var plan=CoinPlanning.MinimumCoins(sizes,target);
if(plan.Count!=2||plan.Coins.Length!=plan.Count||plan.Coins.Any(c=>!sizes.Contains(c))||plan.Coins.Sum()!=target)throw new Exception("Invalid pack plan.");
Console.WriteLine($"minimum packs={plan.Count}, witness={string.Join("+",plan.Coins)}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


A valid witness establishes an upper bound on the optimum; the recurrence proof or an independent exhaustive reference establishes that no better solution exists. Validation and optimality are different obligations.

## Inventory Changes the Answer

```{index} Inventory Changes the Answer
```

An unlimited pack can be reused. A one-use invoice can appear at most once. With two records both worth four, target twelve is feasible with unlimited denomination four but impossible using only those two records.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={4,4};
Console.WriteLine($"unlimited count={CoinPlanning.MinimumCoins(invoices,12).Count}");
Console.WriteLine($"one-use feasible={SubsetPlanning.FindSubset(invoices,12).Indices is not null}");
var eight=SubsetPlanning.FindSubset(invoices,8);
Console.WriteLine($"eight IDs={string.Join(",",eight.Indices!)}, sum={eight.Indices!.Sum(i=>(long)invoices[i])}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}

public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


Duplicate denomination normalization is appropriate for unlimited coins; deduplicating equal-valued invoice positions would lose inventory. The state and input policy follow the business constraints.

## Compare DP with an Independent Small Reference

```{index} Compare DP with an Independent Small Reference
```

Breadth-first exploration of amounts starts at zero and adds one allowed coin at each level. The first discovery of target has the minimum number of coins because every transition costs one. This independent reference can check a small DP instance without reimplementing its amount-order recurrence.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={1,3,4};int target=9;
var distance=new Dictionary<int,int>{{0,0}};var queue=new Queue<int>();queue.Enqueue(0);
while(queue.Count>0)
{
 int amount=queue.Dequeue();
 foreach(int size in sizes)
 {
  int next=amount+size;
  if(next>target||distance.ContainsKey(next))continue;
  distance[next]=distance[amount]+1;queue.Enqueue(next);
 }
}
var plan=CoinPlanning.MinimumCoins(sizes,target);
Console.WriteLine($"DP={plan.Count}, level-search={distance[target]}, witness={string.Join("+",plan.Coins)}");
if(plan.Count!=distance[target])throw new Exception("Reference disagreement.");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


Keep reference targets small and verify unreachable cases as well. Chapter 25 develops breadth-first traversal systematically. Here the level count provides a separate way to validate the minimum-count objective.

## Check Search Against Exhaustive Positions

```{index} Check Search Against Exhaustive Positions
```

Enumerate all subsets of four signed invoice positions. Compare existence with the pruned/memoized search and validate its returned indices and sum. Do not reuse its pruning rule in the reference.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={8,-3,4,4};int target=5;
bool expected=ChoicePlanning.Subsets(invoices.Length).Any(ids=>ids.Sum(i=>(long)invoices[i])==target);
var result=SubsetPlanning.FindSubset(invoices,target);
bool valid=result.Indices is not null&&result.Indices.Distinct().Count()==result.Indices.Length&&result.Indices.All(i=>i>=0&&i<invoices.Length)&&result.Indices.Sum(i=>(long)invoices[i])==target;
if(expected!=(result.Indices is not null)||!valid)throw new Exception("Subset reference disagreement.");
Console.WriteLine($"exhaustive feasible={expected}, witness valid={valid}, IDs={string.Join(",",result.Indices!)}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}

public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


Exhaustive verification is useful on tiny instances; it is not a production solution for large inputs. Check zero targets and empty input separately because an empty witness is a successful result, not a missing one.

## Explain a Recommendation with Evidence

```{index} Explain a Recommendation with Evidence
```

Choose a method from objective, inventory, state range, requested witness, and output volume. Unlimited minimum-pack counts suggest amount DP with predecessor recovery. One-use signed records need a position-aware state; backtracking can stop at the first witness and safely prune signed bounds. Dense manageable state ranges may favor tabulation; sparse visited states may favor memoization.

Report measured calls for a fixed instance alongside worst-case limits. Memoization stores visited states; rolling Fibonacci discards history; materialized permutations retain factorial output. Avoid selecting an algorithm from a keyword in a scenario string. Describe which dependency, constraint, or required result makes the method suitable.

## Exercise

```{index} Exercise
```

Check empty input, zero target, unreachable quantity, and independent witness snapshots. Explain which result means success.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes=Array.Empty<int>();
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sizes=Array.Empty<int>();
var zero=CoinPlanning.MinimumCoins(sizes,0);var missing=CoinPlanning.MinimumCoins(sizes,1);
Console.WriteLine($"zero count={zero.Count}, missing reachable={missing.Count is not null}");
var a=SubsetPlanning.FindSubset(new[]{3,3},6);var b=SubsetPlanning.FindSubset(new[]{3,3},6);a.Indices![0]=99;
Console.WriteLine($"independent witness={string.Join(",",b.Indices!)}");
Console.WriteLine($"empty subset feasible={SubsetPlanning.FindSubset(Array.Empty<int>(),0).Indices is not null}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}

public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


zero count=0, missing reachable=False
independent witness=0,1
empty subset feasible=True


```{rubric} Footnotes
```
[^1]: Match the reference to the exact objective and inventory policy; a reusable-coin oracle does not verify one-use subset search.
[^2]: A search budget is a resource policy. Its exhaustion reports an unknown answer, distinct from proven infeasibility.